# Ecosystem Setup

This notebook walks through the full environment setup for the OpHarvestSeason analysis pipeline.

**This is not required to follow the course.** All results used in the notebooks are precomputed and shipped in `results/`. Run this only if you plan to recompute embeddings/NER yourselves later (e.g. via `scripts/precompute.py` on your own data) and want to check your Ollama + GPU setup works before doing so — it downloads both models (~9GB) and runs a live smoke test.

**What we'll cover:**
1. Python dependencies
2. Ollama models (embeddings + LLM)
3. Verify GPU is being used
4. Smoke-test the full pipeline (parse → embed → query)

## 1. Python Dependencies

In [ ]:
# Install all dependencies
%pip install -q \
    ollama \
    pandas \
    numpy \
    matplotlib \
    seaborn \
    plotly \
    scikit-learn \
    umap-learn \
    langdetect \
    pytz \
    tqdm

## 2. Ollama Models

We use two models:
- **`qwen3-embedding`** — fast, high-quality embeddings. Used for stylometry and cross-forum user correlation.
- **`qwen2.5:14b`** — LLM for language detection, entity extraction, and post classification. 14B fits in 12GB VRAM.

In [ ]:
import subprocess

models = ["qwen3-embedding", "qwen2.5:14b"]

for model in models:
    print(f"Pulling {model}...")
    subprocess.run(["ollama", "pull", model], check=True)
    print(f"  ✓ {model} ready\n")

## 3. Verify GPU Usage

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import ollama

# After pulling a model, check VRAM usage — should show model loaded on GPU
response = ollama.embed(model="qwen3-embedding", input="test")
print(f"Embedding dim: {len(response.embeddings[0])}")

result = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader"], capture_output=True, text=True)
print(f"VRAM used after model load: {result.stdout.strip()}")

## 4. Smoke Test — Full Pipeline

Simulates the real pipeline with synthetic data: parse → embed → compare (cosine similarity).

In [ ]:
import pandas as pd
import ollama
from tqdm import tqdm

# Synthetic forum posts
sample_posts = [
    {"user": "darkuser1", "text": "selling fresh cc dumps, 95% valid rate, pm me"},
    {"user": "darkuser2", "text": "looking for rdp access to us bank networks"},
    {"user": "darkuser1", "text": "new batch available, eu cards, contact via jabber"},
    {"user": "darkuser3", "text": "anyone has working method for cashout 2024?"},
]

df = pd.DataFrame(sample_posts)
df

In [ ]:
# Embed all posts
embeddings = []
for text in tqdm(df["text"], desc="Embedding"):
    response = ollama.embed(model="qwen3-embedding", input=text)
    embeddings.append(response.embeddings[0])

df["embedding"] = embeddings
print(f"Embedded {len(embeddings)} posts — dim={len(embeddings[0])}")

In [ ]:
# Semantic search — brute-force cosine similarity, same approach as scripts/precompute.py.
# No vector DB: at this scale (a handful of posts, or even the full dataset), comparing
# the query against every embedding directly is simpler and fast enough.
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query = "credit card dumps for sale"
query_embedding = ollama.embed(model="qwen3-embedding", input=query).embeddings[0]

matrix = np.array(df["embedding"].tolist())
sims = cosine_similarity([query_embedding], matrix)[0]
top = sims.argsort()[::-1][:3]

print(f"Query: '{query}'\n")
for i in top:
    print(f"  [{df.iloc[i]['user']}] {df.iloc[i]['text']}  (similarity: {sims[i]:.3f})")

## Setup Complete

| Component | Status |
|-----------|--------|
| Python deps | ✓ |
| qwen3-embedding | ✓ |
| qwen2.5:14b | ✓ |
| GPU inference | ✓ |

Ready for case studies.